In [1]:
import sys
print(sys.executable)

c:\Users\st_1o\Desktop\SAC_CAC_Research\.venv\Scripts\python.exe


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


SACの参考資料

https://www.dskomei.com/entry/2022/06/30/222228

In [2]:
!pip install torch

In [3]:
!pip install gymnasium
!pip install gymnasium[classic-control]

  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached farama_notifications-0.0.6-py3-none-any.whl.metadata (729 bytes)
Using cached gymnasium-1.3.0-py3-none-any.whl (953 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached farama_notifications-0.0.6-py3-none-any.whl (2.9 kB)

   -------------------------- ------------- 2/3 [gymnasium]
   ---------------------------------------- 3/3 [gymnasium]

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -------- ------------------------------- 2.1/10.4 MB 39.0 MB/s eta 0:00:01
   ------------------------------------ --- 9.4/10.4 MB 28.0 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 21.0 MB/s  0:00:00


In [3]:
from pathlib import Path #Path クラスは、ファイルシステムへのパスを扱うための便利な方法を提供します。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm  #進捗バーを簡単に追加するための便利なライブラリです。tqdm を使うと、ループ処理や時間のかかる計算タスクの進捗を視覚化することができます。
import seaborn as sns  #データ可視化を行うためのライブラリです。Matplotlibの上に構築されてい
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.distributions import Normal
import gymnasium as gym
import math

In [4]:
import torch

def model_dynamics(x: torch.Tensor, a: torch.Tensor, dt: float = 0.05) -> torch.Tensor:
    """
    Pendulum-v1 の離散時間 Dynamics を分析的に実装。
      x = [cosθ, sinθ, θ̇]
      u = a.squeeze(-1)
      θ̈ = −3g/(2l) sinθ + 3/(m l^2) u
      θ_{t+1} = θ + θ̇·dt
      θ̇_{t+1} = θ̇ + θ̈·dt
    """
    g, l, m = 10.0, 1.0, 1.0
    cos_th, sin_th, thdot = x[:,0], x[:,1], x[:,2]
    th   = torch.atan2(sin_th, cos_th)
    u    = a.squeeze(-1)
    thdd = -3 * g/(2*l) * torch.sin(th) + 3.0/(m*l**2) * u

    th_next    = th + thdot * dt
    thdot_next = thdot + thdd  * dt

    return torch.stack([torch.cos(th_next),
                        torch.sin(th_next),
                        thdot_next],
                       dim=1)


In [5]:
def restricted_direction(
    W1: torch.Tensor,
    W2: torch.Tensor,
    progress: float = 0.0,
    eps: float = 1e-8,
    use_log10_shrink: bool = True,
    log1p_variant: bool = True,
    max_update_norm: float = 4e2,
    debug: bool = False,
    h_value: float = 1.0,     # ← ★追加：安全度 h(s) を渡す
    k: float = 1.63,           # ← ★追加：安全度の鋭さ
    c: float = 0.55            # ← ★追加：安全の閾値
) -> torch.Tensor:
    """
    restricted_direction + 安全度ベースの m_dir 調整（案1）
    h_value: h(s) の平均値（安全度）
    k:       sigmoid の鋭さ
    c:       安全と危険の境界
    """

    device = W1.device
    dtype = W1.dtype

    # --- 既存の安全チェック ---
    if W1.dim() != 1 or W2.dim() != 1:
        raise ValueError("W1 and W2 must be 1D tensors")
    norm1 = W1.norm()
    norm2 = W2.norm()
    if not torch.isfinite(norm1) or not torch.isfinite(norm2):
        return torch.zeros_like(W1)
    if norm1 == 0 and norm2 == 0:
        return torch.zeros_like(W1)
    if norm1 == 0:
        return W2.clone()
    if norm2 == 0:
        return W1.clone()

    # --- W2 の log shrink（既存） ---
    if use_log10_shrink:
        if log1p_variant:
            log_norm2 = torch.log10(norm2 + 1.0)
        else:
            log_norm2 = torch.log10(norm2 + eps)
        log_norm2_pos = torch.clamp(log_norm2, min=0.0)
        shrink = 1.0 / (1.0 + log_norm2_pos)
        W2_use = W2 * shrink
    else:
        W2_use = W2

    # --- m1, m2 の計算（既存） ---
    dot12 = torch.dot(W1, W2_use)
    w1_sq = (norm1 ** 2).clamp(min=eps)
    w2_sq = (norm2 ** 2).clamp(min=eps)

    denom1 = dot12 - w1_sq
    denom1_safe = torch.where(torch.abs(denom1) < eps, torch.sign(denom1) * eps + eps, denom1)
    m1 = dot12 / denom1_safe

    denom2 = w2_sq - dot12
    denom2_safe = torch.where(torch.abs(denom2) < eps, torch.sign(denom2) * eps + eps, denom2)
    m2 = w2_sq / denom2_safe

    # --- 既存の progress ベースのスケジュール ---
    t = float(min(max(progress, 0.0), 1.0))
    m_progress = m2 * (1.0 - t) + m1 * t

    # ============================================================
    # ★★★ ここが案1の核心：安全度ベースの m_dir を導入 ★★★
    # ============================================================

    # h_value が小さい（危険） → safety_level ≈ 1 → m_dir ≈ m1（安全方向）
    # h_value が大きい（安全） → safety_level ≈ 0 → m_dir ≈ m2（性能方向）
    safety_input = torch.tensor(k * (c - h_value), dtype=W1.dtype, device=W1.device)
    safety_level = torch.sigmoid(safety_input)

    # progress と安全度をブレンド
    # safety_level が強いときは安全方向を優先
    m_dir = safety_level * m1 + (1 - safety_level) * m_progress

    # --- m_dir を m1, m2 の範囲にクリップ ---
    m_min = torch.min(m1, m2)
    m_max = torch.max(m1, m2)
    m_dir = torch.clamp(m_dir, min=float(m_min), max=float(m_max))

    # --- 方向ベクトル ---
    w_dir = m_dir * W1 + (1.0 - m_dir) * W2_use

    if w_dir.norm() < eps or not torch.isfinite(w_dir).all():
        return W2_use.clone()

    # --- スケール計算（既存） ---
    w_dir_norm_sq = (w_dir.norm() ** 2).clamp(min=eps)
    m0 = torch.dot(W2_use, w_dir) / w_dir_norm_sq

    m_thresh = 100.0
    p = 1.5
    m0_sign = torch.sign(m0)
    m0_abs = m0.abs()
    m_shrunk = (m0_abs / (1.0 + (m0_abs / m_thresh) ** p)).clamp(min=0.0)
    m_soft = m0_sign * m_shrunk

    W2u_norm = W2_use.norm().clamp(min=1e-6)
    k_rel = 4.5
    rel_cap = k_rel * (W2u_norm + 1e-12)
    m_rel = torch.sign(m_soft) * torch.min(m_soft.abs(), rel_cap)

    M = 21.0
    m_scale = torch.clamp(m_rel, min=-M, max=M)

    w_scaled = m_scale * w_dir

    # --- 最終クリップ ---
    if w_scaled.norm() > max_update_norm:
        w_scaled = w_scaled * (max_update_norm / (w_scaled.norm() + eps))

    if debug:
        print(f"h_value = {h_value}")

    return torch.nan_to_num(w_scaled, nan=0.0), float(m_dir)


In [11]:
def restricted_direction1(
    W1: torch.Tensor,
    W2: torch.Tensor,
    progress: float = 0.0,
    eps: float = 1e-8,
    use_log10_shrink: bool = True,
    log1p_variant: bool = True,
    max_update_norm: float = 4e2,
    debug: bool = False,
    h_value: float = 1.0,     # ← ★追加：安全度 h(s) を渡す
    k: float = 0.8,#1.63,           # ← ★追加：安全度の鋭さ
    c: float = 0.55            # ← ★追加：安全の閾値
) -> torch.Tensor:
    """
    restricted_direction + 安全度ベースの m_dir 調整（案1）
    h_value: h(s) の平均値（安全度）
    k:       sigmoid の鋭さ
    c:       安全と危険の境界
    """

    device = W1.device
    dtype = W1.dtype

    # --- 既存の安全チェック ---
    if W1.dim() != 1 or W2.dim() != 1:
        raise ValueError("W1 and W2 must be 1D tensors")
    norm1 = W1.norm()
    norm2 = W2.norm()
    if not torch.isfinite(norm1) or not torch.isfinite(norm2):
        return torch.zeros_like(W1)
    if norm1 == 0 and norm2 == 0:
        return torch.zeros_like(W1)
    if norm1 == 0:
        return W2.clone()
    if norm2 == 0:
        return W1.clone()

    # --- W2 の log shrink（既存） ---
    if use_log10_shrink:
        if log1p_variant:
            log_norm2 = torch.log10(norm2 + 1.0)
        else:
            log_norm2 = torch.log10(norm2 + eps)
        log_norm2_pos = torch.clamp(log_norm2, min=0.0)
        shrink = 1.0 / (1.0 + log_norm2_pos)
        W2_use = W2 * shrink
    else:
        W2_use = W2

    # --- m1, m2 の計算（既存） ---
    dot12 = torch.dot(W1, W2_use)
    w1_sq = (norm1 ** 2).clamp(min=eps)
    w2_sq = (norm2 ** 2).clamp(min=eps)

    denom1 = dot12 - w1_sq
    denom1_safe = torch.where(torch.abs(denom1) < eps, torch.sign(denom1) * eps + eps, denom1)
    m1 = dot12 / denom1_safe

    denom2 = w2_sq - dot12
    denom2_safe = torch.where(torch.abs(denom2) < eps, torch.sign(denom2) * eps + eps, denom2)
    m2 = w2_sq / denom2_safe

    # --- 既存の progress ベースのスケジュール ---
    t = float(min(max(progress, 0.0), 1.0))
    m_progress = m1 * (1.0 - t) + m2 * t

    # ============================================================
    # ★★★ ここが案1の核心：安全度ベースの m_dir を導入 ★★★
    # ============================================================
    # m1 が表す方向（W1 に直交するところで W2 寄り）
    dir1 = m1 * W1 + (1.0 - m1) * W2_use

    # m2 が表す方向（W2 に直交するところで W1 寄り）
    dir2 = m2 * W1 + (1.0 - m2) * W2_use

    # h_value が小さい（危険） → safety_level ≈ 1 → m_dir ≈ m1（安全方向）
    # h_value が大きい（安全） → safety_level ≈ 0 → m_dir ≈ m2（性能方向）
    safety_input = torch.tensor(k * (c - h_value), dtype=W1.dtype, device=W1.device)
    safety_level = torch.sigmoid(safety_input)

    # progress と安全度をブレンド
    # safety_level が強いときは安全方向を優先
    #m_dir = safety_level * m2 + (1 - safety_level) * m_progress
    m_dir = (1.0 - safety_level) * dir1 + safety_level * dir2

    # --- m_dir を m1, m2 の範囲にクリップ ---
    #m_min = torch.min(m1, m2)
    #m_max = torch.max(m1, m2)
    #m_dir = torch.clamp(m_dir, min=float(m_min), max=float(m_max))

    # --- 方向ベクトル ---
    #w_dir = m_dir * W1 + (1.0 - m_dir) * W2_use

    if m_dir.norm() < eps or not torch.isfinite(m_dir).all():
        return W2_use.clone()

    # --- スケール計算（既存） ---
    m_dir_norm_sq = (m_dir.norm() ** 2).clamp(min=eps)
    m0 = torch.dot(W2_use, m_dir) / m_dir_norm_sq

    m_thresh = 100.0
    p = 1.5
    m0_sign = torch.sign(m0)
    m0_abs = m0.abs()
    m_shrunk = (m0_abs / (1.0 + (m0_abs / m_thresh) ** p)).clamp(min=0.0)
    m_soft = m0_sign * m_shrunk

    W2u_norm = W2_use.norm().clamp(min=1e-6)
    k_rel = 4.5
    rel_cap = k_rel * (W2u_norm + 1e-12)
    m_rel = torch.sign(m_soft) * torch.min(m_soft.abs(), rel_cap)

    M = 21.0
    m_scale = torch.clamp(m_rel, min=-M, max=M)

    w_scaled = m_scale * m_dir

    # --- 最終クリップ ---
    if w_scaled.norm() > max_update_norm:
        w_scaled = w_scaled * (max_update_norm / (w_scaled.norm() + eps))

    if debug:
        print(f"h_value = {h_value}")

    return torch.nan_to_num(w_scaled, nan=0.0), m_dir.clone()

In [6]:
gym_name = 'Pendulum-v1'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 123456
torch.manual_seed(seed)
np.random.seed(seed)


In [8]:
class ClippedCriticNet(nn.Module):

    def __init__(self, input_num, output_num, hidden_size):

        super().__init__()

        self.linear1 = nn.Linear(input_num, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, output_num)

        self.linear4 = nn.Linear(input_num, hidden_size)
        self.linear5 = nn.Linear(hidden_size, hidden_size)
        self.linear6 = nn.Linear(hidden_size, output_num)

    def forward(self, state, action):
        xu = torch.cat([state, action], 1)

        x1 = F.relu(self.linear1(xu)) ##network1
        x1 = F.relu(self.linear2(x1))
        x1 = self.linear3(x1)

        x2 = F.relu(self.linear4(xu)) ##network2
        x2 = F.relu(self.linear5(x2))
        x2 = self.linear6(x2)

        return x1, x2  #network1, network2の出力

Actor では、ネットワークの出力の際にエントロピー項を追加しています。この値が方策の更新時と Soft Q 関数の更新時の損失値を求めるために使われます。エントロピー項には、出力値の対数確率に
−log(1−y2)+ε
 を足しています。これは、出力値が上下限に張り付かないようにしており、探索範囲の拡大に寄与しています。

In [9]:
class SoftActorNet(nn.Module):

    def __init__(self, input_num, output_num, hidden_size, action_scale):

        super().__init__()

        self.linear1 = nn.Linear(input_num, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)

        self.mean_linear = nn.Linear(hidden_size, output_num)
        self.log_std_linear = nn.Linear(hidden_size, output_num)

        self.action_scale = torch.tensor(action_scale)
        self.action_bias = torch.tensor(0.)

    def forward(self, state, LOG_SIG_MAX = 2, LOG_SIG_MIN = -20):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, min=LOG_SIG_MIN, max=LOG_SIG_MAX)  #PyTorchでテンソルの値を指定した範囲に制限するための関数です。
        return mean, log_std   #出力は平均と（対数にした）分散

    def sample(self, state,epsilon = 1e-6):
        self.epsilon=epsilon
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = Normal(mean, std)  #平均mean、標準偏std2の正規分布を定義
        x_t = normal.rsample()      #rsample() と sample() の違いは、rsample() はサンプリングの際に勾配追跡が可能になる点です（requires_grad=True のテンソルに対して有効）
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)       #与えられた値がその分布に従う確率密度関数（PDF）の対数値を計算します。
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + self.epsilon)
            #y はネットワークの出力値で、通常 tanh 関数を使用して制限されています。
            #log(1 − y²) は、tanh の特性を利用し、出力値が端（±1付近）に近いほど大きなペナルティを与える形で計算されます。
            #ε は、小さい定数であり、ゼロ除算を回避するために加えられます。

        log_prob = log_prob.sum(1, keepdim=True)    #テンソルの指定された次元で要素を加算する際に使用されるメソッドです。
                                                    #特に、keepdim=True を設定すると、計算後のテンソルの形状が元の次元を保持します（
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean



    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super().to(device)



これらのネットワークを使って SAC モデルを設計します。ポイントになるのは、「update_parameters 関数」内での Critic と Actor の損失値を求めるところです。両方ともエントロピー項を加味した計算になっています。そして、最後に
α
 を最適化しています。

In [ ]:
class SoftActorCriticModel(object):

    def __init__(self, state_num, action_num, action_scale, args, device):
        self.args = args

        self.h_buffer = []   # ★追加：安全度の移動平均用バッファ
        self.h_buffer_size = 80  # ★追加：移動平均の窓幅（推奨20）

        self.m_dir_buffer = []


        self.gamma = args['gamma']
        self.tau = args['tau']
        self.alpha = args['alpha']
        self.device = device
        self.target_update_interval = args['target_update_interval']
        self.updates = 0

                # 履歴記録（エピソード毎）
        self.history_safe_grad_norms = []      # safety 勾配ノルム（エピソード単位の平均または最終値）
        self.history_stability_grad_norms = [] # stability 勾配ノルム
        self.history_safe_grad_norms_stage1 = []   # ステージ1 用（オプション）
        self.history_stability_grad_norms_stage2 = [] # ステージ2 用（オプション）

        # ステージ２用の安全項スケールを args に追加しておく
        self.lambda_safe = args.get('lambda_safe', 4.0)

        self.actor_net = SoftActorNet(
            input_num=state_num, output_num=action_num, hidden_size=args['hidden_size'], action_scale=action_scale).to(self.device)

        self.critic_net = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(device=self.device)

        self.critic_net_target = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(self.device)

        hard_update(self.critic_net_target, self.critic_net)
        convert_network_grad_to_false(self.critic_net_target)

        self.actor_optim = optim.Adam(self.actor_net.parameters(),lr=1e-4)
        self.critic_optim = optim.Adam(self.critic_net.parameters(),lr=args.get('critic_lr', 3e-4))

        self.critic_safe        = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(device=self.device)
        self.critic_safe_target = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(self.device)
        hard_update(self.critic_safe_target, self.critic_safe)
        convert_network_grad_to_false(self.critic_safe_target)
        self.critic_safe_optim  = optim.Adam(self.critic_safe.parameters(),lr=5e-5)


        #self.target_entropy = -torch.prod(torch.Tensor(action_num).to(self.device)).item()  #target entropy H_tは、行動の次元数dに対して、  H_t=-d
                    #torch.prod関数を使用して、テンソル内の全要素の積を計算し、その結果をPythonの数値型（item()）に変換する
        self.target_entropy = -float(action_num)
        self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device)  #torch.zeros() を使用して 値がすべて0のテンソル を作成しています。1要素だけのテンソルを作成します。
        self.alpha_optim = optim.Adam([self.log_alpha])   #log_alphaのパラメータを最適化するためのAdamオプティマイザーを初期化します。

    def select_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        if not evaluate:
            action, _, _ = self.actor_net.sample(state)
        else:
            _, _, action = self.actor_net.sample(state)
        return action.cpu().detach().numpy().reshape(-1)

    def h(self, x: torch.Tensor) -> torch.Tensor:
        thdot = x[:, 2]
        h_raw = thdot + 2.0              # θ̇ = -2 → h_raw = 0（境界）
        # 正の側だけクリップ、負の側はそのまま
        h_pos_clipped = torch.clamp(h_raw, max=2.0)  # 上だけ 2 に飽和
        h_norm = h_pos_clipped / 2.0                 # 安全側は最大 1
        return h_norm

    def h_raw(self, s):
        # Pendulum の角速度は s[:,2]
        thdot = s[:, 2]
        return thdot + 2.0   # ← 生の h(s)




    def compute_safety_reward(self, s, a, s_next, alpha):
        """
        Control Barrier Function に基づく瞬時安全報酬 r_safe
        r_safe = exp(min(h(s') + (γ0-1)*h(s), 0))
        """
        h_s      = self.h(s)
        h_s_next = self.h(s_next)
        raw      = h_s_next + (alpha - 1.0) * h_s
        clipped = torch.clamp(raw, max=0.0)
        return torch.exp(clipped) - 1.0

    def compute_nav_reward(self, s, a, s_next):
        """
        既存の安定／目標達成報酬 r_nav を返すラッパー
        """
        return self.env_reward(s, a, s_next)

    def update_critics_and_actor(self, batch, episode_index):
        """
        Stage1/Stage2 の切り替えを含む
        - batch: replay buffer から取ってきたミニバッチ
        - episode_index: 現在のエピソード番号（1始まり）
        """
        self.updates += 1

        max_grad_norm = 500.0

        stage1 = False#(episode_index <= args['stage1_episodes'])

        # 1) バッチの展開
        s, a, reward_tuple, s_next, mask = batch
        s      = s.to(self.device)
        a      = a.to(self.device)
        s_next = s_next.to(self.device)
        mask   = mask.to(self.device)
        # (必要に応じて mask, logp_old を batch に含めてください)
        reward_tensor = torch.FloatTensor(reward_tuple).to(self.device)
        r_nav, r_safe = reward_tensor[:, 0].unsqueeze(1), \
                                 reward_tensor[:, 1].unsqueeze(1)

        # 2) Critic 更新（常に行う）
        with torch.no_grad():
            a_next, logp_next, _ = self.actor_net.sample(s_next)
            q1_t, q2_t = self.critic_net_target(s_next, a_next)
            q_min     = torch.min(q1_t, q2_t) - self.alpha * logp_next
            target_nav  = r_nav  + mask.unsqueeze(1) * self.gamma * q_min
            # 安全性クリティックのターゲット
            qs1_t, qs2_t = self.critic_safe_target(s_next, a_next)
            qs_min       = torch.min(qs1_t, qs2_t)
            target_safe  = r_safe + mask.unsqueeze(1) * self.gamma * qs_min

        if not stage1:
            q1, q2   = self.critic_net(s, a)
            loss_nav = F.mse_loss(q1, target_nav) + F.mse_loss(q2, target_nav)
            q_mean   = torch.min(q1, q2).mean().item()
        else:
            loss_nav = None  # Stage1 は CriticNav 更新をスキップ
            q_mean   = None

        qs1, qs2   = self.critic_safe(s, a)
        loss_safe  = F.mse_loss(qs1, target_safe) + F.mse_loss(qs2, target_safe)

        # Critic
        if loss_nav is not None:
            self.critic_optim.zero_grad()

            loss_nav.backward()
            # <<< INSERT: global grad clip for critic >>下２行
            max_grad_norm = 500.0
            torch.nn.utils.clip_grad_norm_(self.critic_net.parameters(), max_grad_norm)

            self.critic_optim.step()

        self.critic_safe_optim.zero_grad()

        loss_safe.backward()
        # <<< INSERT: global grad clip for critic_safe >>下１行
        torch.nn.utils.clip_grad_norm_(self.critic_safe.parameters(), max_grad_norm)

        self.critic_safe_optim.step()

        # --- 安全クリティック勾配ノルムを計算して保持 ---
        total_sq = 0.0
        for p in self.critic_safe.parameters():
            if p.grad is not None:
                total_sq += float(p.grad.data.norm(2).item()) ** 2 #p.grad.data.norm(2).item() ** 2　変更したよ
        critic_safe_grad_norm = total_sq ** 0.5 #追加したよ
        self.last_safe_grad_norm = critic_safe_grad_norm #追加したよ


        # 3) Actor 更新
        # stability_loss と safety_loss を定義
        pi, logp_pi, _ = self.actor_net.sample(s)
        q1_pi, q2_pi   = self.critic_net(s, pi)
        q_min_pi       = torch.min(q1_pi, q2_pi)



        stability_loss = (self.alpha * logp_pi - q_min_pi).mean()

        # safety_loss
        qs1_pi, qs2_pi    = self.critic_safe(s, pi)
        safety_loss_unscaled = - torch.min(qs1_pi, qs2_pi).mean()
        safety_loss = self.lambda_safe * safety_loss_unscaled
        #safety_loss_stage = - self.lambda_safe * torch.min(qs1_pi, qs2_pi).mean()



        # Stage判定
        #stage1 = (episode_index <= args['stage1_episodes'])

        if episode_index <= args['stage1_episodes']:
            # ステージ1：安全性のみで Actor 更新

            self.actor_optim.zero_grad()

            safety_loss.backward()
            self.actor_optim.step()

            nav_loss_item   = None
            safe_loss_item  = safety_loss.item()

        else:
            # ステージ2：restricted_direction を用いた制限付き更新
            # a) stability勾配
            self.actor_optim.zero_grad()

            stability_loss.backward(retain_graph=True)
            grad_st = torch.cat([p.grad.view(-1) for p in self.actor_net.parameters()])

            # b) safety勾配

            self.actor_optim.zero_grad()

            safety_loss.backward(retain_graph=True)
            grad_sa = torch.cat([p.grad.view(-1) for p in self.actor_net.parameters()])

            # 生のノルムを記録（正規化前の大きさ）
            st_norm_raw = grad_st.norm().item()
            sa_norm_raw = grad_sa.norm().item()
            self.history_stability_grad_norms.append(st_norm_raw)
            self.history_safe_grad_norms.append(sa_norm_raw)


            # d) 正規化 ＆ restricted_direction

            stage1_eps = float(self.args.get('stage1_episodes', 0))
            stage2_eps = float(self.args.get('stage2_episodes', 1))
            """if episode_index <= stage1_eps:
                progress = 0.0
            else:
                progress = min(1.0, max(0.0, (episode_index - stage1_eps) / max(1.0, stage2_eps)))"""

            progress = min(1.0, (episode_index / args['stage2_episodes'])**2)

            #e = restricted_direction(grad_sa, grad_st,progress=progress, debug=False)
            # --- h(s) をバッチ単位で取得 ---
            with torch.no_grad():
                h_batch = self.h(s)  # shape: [batch_size]

            # --- 危ない方（下位15%）を代表値として採用 ---
            h_value_batch = float(torch.quantile(h_batch, 0.15).item())
            #h_s = agent.h(s).mean().item()#ここから
            # --- 移動平均バッファに追加 ---
            self.h_buffer.append(h_value_batch)
            if len(self.h_buffer) > self.h_buffer_size:
                self.h_buffer.pop(0)

            # --- 移動平均を計算 ---
            h_value = float(np.mean(self.h_buffer))
            e, m_dir = restricted_direction(grad_sa, grad_st, progress=progress, h_value=h_value)#ここまで
            self.last_m_dir = m_dir

            dot   = torch.dot(grad_st, grad_sa).item()
            norm1 = grad_st.norm().item()
            norm2 = grad_sa.norm().item()
            cos_sim = dot / (norm1 * norm2 + 1e-8)

            angle = math.degrees(math.acos(dot/(norm1*norm2+1e-8)))
            #print(f"[DEBUG] dot(W_nav,W_safe)={dot:.3f},cos={cos_sim:.3f}, angle={angle:.1f}°")


        # 展開前に p.grad を上書きするため、ここでは展開後に出力する

            # after building e and before assigning to p.grad (or right after assignment but before optimizer.step)
            max_e_norm = 102.5   # safety bound on the norm of the assembled gradient vector
            e_norm = e.norm().item()
            if e_norm > max_e_norm:
                e = e * (max_e_norm / (e_norm + 1e-12))
            # then write back to p.grad as before and run actor step上４行追加したよ

            # e) 各パラメータ勾配に展開
            idx = 0
            for p in self.actor_net.parameters():
                n = p.numel()
                p.grad = e[idx:idx+n].view_as(p).clone()
                idx += n

            max_grad_norm = 50.0#下２行
            torch.nn.utils.clip_grad_norm_(self.actor_net.parameters(), max_grad_norm)

            # f) 更新
            self.actor_optim.step()

            nav_loss_item  = stability_loss.item()
            safe_loss_item = safety_loss.item()

        # 4) Entropy α 更新（既存ロジック）
        alpha_loss = -(self.log_alpha * (logp_pi + self.target_entropy).detach()).mean()
        self.alpha_optim.zero_grad()
        alpha_loss.backward()
        self.alpha_optim.step()
        self.alpha = self.log_alpha.exp()

        # 5) ターゲットネットワークのソフト更新
        if self.updates % self.target_update_interval == 0:
            soft_update(self.critic_net_target, self.critic_net, self.tau)
            soft_update(self.critic_safe_target, self.critic_safe, self.tau)

                # === ノルム記録 ===
        # stage1 の場合は安全クリティックの勾配ノルム（self.last_safe_grad_norm が既に保持されている）
        # stage2 の場合は stability と safety 勾配の L2 ノルム（計算済み grad_st, grad_sa を使う）
        try:
            if stage1:
                # stage1: 安全クリティック勾配ノルム（CriticSafe の勾配ノルム）
                self.history_safe_grad_norms.append(self.last_safe_grad_norm if hasattr(self, 'last_safe_grad_norm') else 0.0)
                # stability はこの段階では未更新なので 0 を格納
                self.history_stability_grad_norms.append(0.0)
            else:
                # stage2: actor の stability/safety 勾配ノルム（grad_st, grad_sa は正規化前の値を使うのが望ましい）
                # grad_st, grad_sa はスコープ内で定義されているはずなので取得する
                st_norm = grad_st.norm().item() if 'grad_st' in locals() else 0.0
                sa_norm = grad_sa.norm().item() if 'grad_sa' in locals() else 0.0
                self.history_stability_grad_norms.append(st_norm)
                self.history_safe_grad_norms.append(sa_norm)
        except Exception:
            # 記録でエラーが出ても学習を止めない
            self.history_safe_grad_norms.append(0.0)
            self.history_stability_grad_norms.append(0.0)


        # stage1 の場合 grad_st/grad_sa/dot は未定義の可能性があるため安全値を返す
        try:
            st_norm_raw = st_norm_raw if 'st_norm_raw' in locals() else 0.0
            sa_norm_raw = sa_norm_raw if 'sa_norm_raw' in locals() else self.last_safe_grad_norm if hasattr(self, 'last_safe_grad_norm') else 0.0
            dot_raw     = dot if 'dot' in locals() else 0.0
        except Exception:
            st_norm_raw, sa_norm_raw, dot_raw = 0.0, 0.0, 0.0

        return nav_loss_item, safe_loss_item, critic_safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw



In [11]:
def soft_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(target_param.data * (1.0 - tau) + param.data * tau)


def hard_update(target, source):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(param.data)


def convert_network_grad_to_false(network):
    for param in network.parameters():
        param.requires_grad = False

In [ ]:
import random


class ReplayMemory:

    def __init__(self, memory_size):
        self.memory_size = memory_size
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, mask):
        if len(self.buffer) < self.memory_size:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, mask)
        self.position = (self.position + 1) % self.memory_size

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        state      = torch.from_numpy(state).float()
        action     = torch.from_numpy(action).float()
        reward     = torch.from_numpy(reward).float()
        next_state = torch.from_numpy(next_state).float()
        done       = torch.from_numpy(done).float()
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

    def clear(self):
        # ここを追加
        self.buffer.clear()
        self.position = 0

In [ ]:
import os
import time
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42


args = {
    'gym_name' : 'Pendulum-v1',
    'gamma': 0.99,
    'tau': 0.005,
    'alpha': 0.9,
    'seed': 123456,
    'batch_size': 256,
    'hidden_size': 256,
    'start_steps': 1000,
    'updates_per_step': 1,
    'target_update_interval': 1,
    'memory_size': 100000,
    'epochs': 100,
    'eval_interval': 10,
    'stage1_episodes': 0,
    'stage2_episodes': 250,
    'log_interval':  10,
}

BASE_SEED = 123456  # run_seed = BASE_SEED + run_id (per-run reproducible seeding)

total_episodes = args['stage1_episodes'] + args['stage2_episodes']

# ============================================================
# 1. 1回分のデータ保存関数
# ============================================================

def save_single_run(run_id, save_dir="runs"):
    os.makedirs(save_dir, exist_ok=True)

    data = {
        "episode_rewards_nav": np.array(episode_rewards_nav),
        "episode_rewards_safe": np.array(episode_rewards_safe),
        "episode_h": np.array(episode_h),
        "episode_h_value": np.array(episode_h_value),
        "episode_m_dir": np.array(episode_m_dir),
        "episode_violations": np.array(episode_violations),
        "episode_safe_grad_stage2": np.array(episode_safe_grad_stage2),
        "episode_stability_grad": np.array(episode_stability_grad),
        "episode_grad_dot": np.array(episode_grad_dot),
        "episode_theta_dot_mean": np.array(episode_theta_dot_mean),
        "episode_theta_dot_max": np.array(episode_theta_dot_max),
        "episode_theta_dot_min": np.array(episode_theta_dot_min),
        "runtime_sec": time.time() - start_time,
    }

    np.savez_compressed(f"{save_dir}/run_{run_id}.npz", **data)
    print(f"[SAVE] Saved run data to {save_dir}/run_{run_id}.npz")


# ============================================================
# 2. 図を保存する関数
# ============================================================

def save_figures_for_run(run_id, save_dir="runs"):
    os.makedirs(save_dir, exist_ok=True)

    # 1. 安全報酬
    plt.figure(figsize=(10,4))
    plt.plot(episode_rewards_safe)
    plt.ylim(-200, 0)
    plt.title(f"Safe Reward (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_safe_reward.png")
    plt.close()

    # 2. 性能報酬
    plt.figure(figsize=(10,4))
    plt.plot(episode_rewards_nav)
    plt.ylim(-2000, 0)
    plt.title(f"Navigation Reward (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_nav_reward.png")
    plt.close()


    # 3. m_dir
    plt.figure(figsize=(10,4))
    plt.plot(episode_m_dir)
    plt.title(f"m_dir (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_m_dir.png")
    plt.close()

    # 4. h_value
    plt.figure(figsize=(10,4))
    plt.plot(episode_h_value)
    plt.title(f"h_value (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_h_value.png")
    plt.close()

    # 5. h(s)
    plt.figure(figsize=(10,4))
    plt.plot(episode_h)
    plt.title(f"h(s) (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_h_s.png")
    plt.close()

    # 6. 安全違反
    plt.figure(figsize=(10,4))
    plt.plot(episode_violations)
    plt.title(f"Violations (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_violations.png")
    plt.close()

    # 7. grad_dot
    plt.figure(figsize=(10,4))
    plt.plot(episode_grad_dot)
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f"grad_st · grad_sa (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_grad_dot.png")
    plt.close()

    # 8. stability_grad
    plt.figure(figsize=(10,4))
    plt.plot(episode_stability_grad)
    plt.title(f"Stability Gradient Norm (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_stability_grad.png")
    plt.close()

    # 9. safe_grad
    plt.figure(figsize=(10,4))
    plt.plot(episode_safe_grad_stage2)
    plt.title(f"Safe Gradient Norm (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_safe_grad.png")
    plt.close()

    # 10. theta_dot
    plt.figure(figsize=(10,4))
    plt.plot(episode_theta_dot_mean, label="mean")
    plt.plot(episode_theta_dot_max, label="max")
    plt.plot(episode_theta_dot_min, label="min")
    plt.legend()
    plt.title(f"Angular Velocity Stats (Run {run_id})")
    plt.grid()
    plt.savefig(f"{save_dir}/run_{run_id}_theta_dot.png")
    plt.close()

    print(f"[SAVE] Saved ALL figures for run {run_id}")
    

def run_experiment(run_id):

    print(f"\n===== RUN {run_id} START =====")
    # -----------------------------
    # 1. すべての episode_xxx を初期化
    # -----------------------------
    global episode_rewards_nav, episode_rewards_safe
    global episode_h, episode_h_value, episode_m_dir
    global episode_violations, episode_safe_grad_stage2
    global episode_stability_grad, episode_grad_dot
    global episode_theta_dot_mean, episode_theta_dot_max, episode_theta_dot_min
    global n_steps, n_update, printed_stage1, start_time

    episode_rewards_nav = []
    episode_rewards_safe = []
    episode_h = []
    episode_h_value = []
    episode_m_dir = []
    episode_violations = []
    episode_safe_grad_stage2 = []
    episode_stability_grad = []
    episode_grad_dot = []
    episode_theta_dot_mean = []
    episode_theta_dot_max = []
    episode_theta_dot_min = []


    # -----------------------------
    # 2. 環境・エージェント・メモリを毎回作り直す（重要）
    # -----------------------------
    run_seed = BASE_SEED + run_id
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)

    env = gym.make(args['gym_name'])
    env.action_space.seed(run_seed)

    agent = SoftActorCriticModel(
        state_num=env.observation_space.shape[0],
        action_num=env.action_space.shape[0],
        action_scale=env.action_space.high[0],
        args=args,
        device=device
    )

    for opt in (agent.actor_optim, agent.critic_optim, agent.critic_safe_optim, agent.alpha_optim):#下３行追加したよー
        for g in opt.param_groups:
            g['lr'] = g.get('lr', 3e-4) * 0.3#上３行はつかう 1e-3

    """for g in agent.alpha_optim.param_groups:
           g['lr'] *= 0.5 """

    # Inserted: lower critic_safe lr to base_lr * 0.125 (absolute overwrite)
    base_lr = 2e-4
    for g in agent.critic_safe_optim.param_groups:
        g['lr'] = base_lr * 0.75#0.50 #0.25 #0.125

    memory = ReplayMemory(args['memory_size'])

    # -----------------------------
    # 3. カウンタ初期化
    # -----------------------------
    n_steps = 0
    n_update = 0
    printed_stage1 = False
    start_time = time.time()


    # ★★★ ここにあなたの「学習ループ（for ep in ...）」を丸ごと入れる ★★★
    
    for ep in range(1, total_episodes + 1):
    # === α を小 → 大にスケジューリング ===
        alpha_start = 0.1
        alpha_end   = 0.9
        total_eps   = total_episodes

        new_alpha = alpha_start + (alpha_end - alpha_start) * (ep / total_eps)
        args['alpha'] = new_alpha
        agent.alpha = torch.tensor(new_alpha).to(device)

        if ep % 5 == 0:
            print(f"[Alpha Schedule] ep={ep}, alpha={new_alpha:.4f}")

        # -----------------------------
        # エピソード初期化
        # -----------------------------

        ep_ret_safe = 0.0
        ep_ret_nav = 0.0

        perupdate_st_norms = []
        perupdate_sa_norms = []
        perupdate_dots = []
        perupdate_safe_grad_norms = []


        perupdate_ratio_m_list = [] 
        loss_nav_list = []
        loss_safe_list = []

        violations    = 0
        done = False
        if ep == 1:
            state, _ = env.reset(seed=run_seed)
        else:
            state, _ = env.reset()

        # [ADDED START] per-episode theta_dot collection
        theta_dot_vals = []
        theta_dot_signed = []       # stores signed theta_dot for this episode
        # [ADDED END]

        perupdate_m_dir = []
        perstep_h = []   # ← h(x) を保存するバッフ

        while not done:

            if args['start_steps'] > n_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state)

            # === h(x) を保存 ===
            s_t = torch.from_numpy(state).float().to(device).unsqueeze(0)
            h_val = agent.h(s_t).item()
            perstep_h.append(h_val)


            if len(memory) > args['batch_size']:
                for _ in range(args['updates_per_step']):
                    batch = memory.sample(args['batch_size'])

                    res = agent.update_critics_and_actor(batch, episode_index=ep)


                    if res is None:
                        loss_nav, loss_safe, safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw = (None, None, None, None, None, None)
                    else:
                        loss_nav, loss_safe, safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw = res
                
                    if hasattr(agent, "last_m_dir"):
                        perupdate_m_dir.append(agent.last_m_dir)



                    # original loss list append
                    if loss_safe is not None:########
                        loss_safe_list.append(loss_safe)##########
                    if loss_nav is not None:
                        loss_nav_list.append(loss_nav)

                    # append per-update metrics into the episode-local buffers
                    if safe_grad_norm is not None:#safe_grad_normから変更追加した
                        perupdate_safe_grad_norms.append(float(safe_grad_norm))#ここも同じ変更
                    if st_norm_raw is not None:
                        perupdate_st_norms.append(float(st_norm_raw))
                    if sa_norm_raw is not None:
                        perupdate_sa_norms.append(float(sa_norm_raw))
                    if dot_raw is not None:
                        perupdate_dots.append(float(dot_raw))
                        
                    n_update += 1

            # 環境ステップ
            next_state, r_env, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            state_t      = torch.from_numpy(state).float().to(device)
            next_state_t = torch.from_numpy(next_state).float().to(device)
            

            #安全報酬
            action_t = torch.from_numpy(action).float().to(device).unsqueeze(0)
            r_safe = agent.compute_safety_reward(
                 state_t.unsqueeze(0),
                 action_t,
                 next_state_t.unsqueeze(0),
                 args['alpha']
             ).item()
             # --- Barrier 関数違反カウント ---
            # h(s_next)<0 を「違反」としてカウント
            if agent.h(next_state_t.unsqueeze(0)).item() < 0.0:
                violations += 1

             # 両者をタプルで保存
            memory.push(state=state, action=action,reward=(r_env, r_safe), next_state=next_state,mask = float(not done))

            # [ADDED START] collect theta_dot (Pendulum obs: [cos, sin, theta_dot])
            theta_dot = float(next_state[2])
            theta_dot_signed.append(theta_dot)           # record signed value
            theta_dot_vals.append(abs(theta_dot))  # store absolute angular velocity
            # [ADDED END]


            #else:
            #  done=False
            n_steps += 1
            ep_ret_nav += r_env
            ep_ret_safe+= r_safe

            state = next_state
        # -----------------------------
        # エピソード終了処理
        # -----------------------------
        episode_rewards_nav.append(ep_ret_nav)
        episode_rewards_safe.append(ep_ret_safe)
        episode_violations.append(violations)

        episode_h.append(float(np.mean(perstep_h)) if perstep_h else 0.0)
        episode_h_value.append(float(np.mean(agent.h_buffer)) if hasattr(agent,"h_buffer") and agent.h_buffer else 0.0)
        episode_m_dir.append(float(np.mean(perupdate_m_dir)) if perupdate_m_dir else 0.0)
        


        if theta_dot_vals:
            episode_theta_dot_mean.append(float(np.mean(theta_dot_vals)))
            episode_theta_dot_max.append(float(np.max(theta_dot_vals)))
            episode_theta_dot_min.append(float(np.min(theta_dot_vals)))
        else:
            episode_theta_dot_mean.append(0.0)
            episode_theta_dot_max.append(0.0)
            episode_theta_dot_min.append(0.0)

        # 勾配統計
        if perupdate_safe_grad_norms:
            episode_safe_grad_stage2.append(float(np.mean(perupdate_safe_grad_norms)))
        else:
            episode_safe_grad_stage2.append(0.0)

        if perupdate_st_norms:
            episode_stability_grad.append(float(np.mean(perupdate_st_norms)))
        else:
            episode_stability_grad.append(0.0)

        if perupdate_dots:
            episode_grad_dot.append(float(np.mean(perupdate_dots)))
        else:
            episode_grad_dot.append(0.0)
            
        
    # つまり、今あなたが貼った巨大な for ep in ... の部分を
    # そのまま run_experiment() の中に移動させる

    # --- 3. 1回分のデータ保存 ---
    save_single_run(run_id)
    save_figures_for_run(run_id)

    print(f"===== RUN {run_id} END =====\n")


In [21]:
NUM_RUNS = 10
for run_id in range(1, NUM_RUNS + 1):
    run_experiment(run_id)



===== RUN 1 START =====
[Alpha Schedule] ep=5, alpha=0.1160
[Alpha Schedule] ep=10, alpha=0.1320
[Alpha Schedule] ep=15, alpha=0.1480
[Alpha Schedule] ep=20, alpha=0.1640
[Alpha Schedule] ep=25, alpha=0.1800
[Alpha Schedule] ep=30, alpha=0.1960
[Alpha Schedule] ep=35, alpha=0.2120
[Alpha Schedule] ep=40, alpha=0.2280
[Alpha Schedule] ep=45, alpha=0.2440
[Alpha Schedule] ep=50, alpha=0.2600
[Alpha Schedule] ep=55, alpha=0.2760
[Alpha Schedule] ep=60, alpha=0.2920
[Alpha Schedule] ep=65, alpha=0.3080
[Alpha Schedule] ep=70, alpha=0.3240
[Alpha Schedule] ep=75, alpha=0.3400
[Alpha Schedule] ep=80, alpha=0.3560
[Alpha Schedule] ep=85, alpha=0.3720
[Alpha Schedule] ep=90, alpha=0.3880
[Alpha Schedule] ep=95, alpha=0.4040
[Alpha Schedule] ep=100, alpha=0.4200
[Alpha Schedule] ep=105, alpha=0.4360
[Alpha Schedule] ep=110, alpha=0.4520
[Alpha Schedule] ep=115, alpha=0.4680
[Alpha Schedule] ep=120, alpha=0.4840
[Alpha Schedule] ep=125, alpha=0.5000
[Alpha Schedule] ep=130, alpha=0.5160
[Alpha 

In [ ]:
import glob

def load_all_runs(folder="runs"):
    files = sorted(glob.glob(f"{folder}/run_*.npz"))
    nav_list = []
    safe_list = []
    h_list = []
    hval_list = []
    mdir_list = []
    dot_list = []
    viol_list = []
    runtime_list = []

    for f in files:
        data = np.load(f)
        nav_list.append(data["episode_rewards_nav"])
        safe_list.append(data["episode_rewards_safe"])
        h_list.append(data["episode_h"])
        hval_list.append(data["episode_h_value"])
        mdir_list.append(data["episode_m_dir"])
        dot_list.append(data["episode_grad_dot"])
        viol_list.append(data["episode_violations"])
        runtime_list.append(float(data["runtime_sec"]))

    return (np.array(nav_list), np.array(safe_list),
            np.array(h_list), np.array(hval_list),
            np.array(mdir_list), np.array(dot_list),
            np.array(viol_list), np.array(runtime_list),
            )


In [ ]:
(nav_all, safe_all, h_all, hval_all, mdir_all, dot_all, viol_all, runtimes) = load_all_runs()

n_runs = nav_all.shape[0]  # 実際にロードされたrun数（ハードコードせず動的に取得）

nav_mean = nav_all.mean(axis=0)
nav_std  = nav_all.std(axis=0)

safe_mean = safe_all.mean(axis=0)
safe_std  = safe_all.std(axis=0)

# --- 性能報酬 ---
plt.figure(figsize=(10,4))
plt.plot(nav_mean, label="mean nav reward")
plt.fill_between(range(len(nav_mean)),
                 nav_mean - nav_std,
                 nav_mean + nav_std,
                 alpha=0.3)
plt.ylim(-2000, 0)
plt.title(f"Navigation Reward (mean ± std over {n_runs} runs)")
plt.grid()
plt.legend()
plt.savefig("nav_mean_std.png", dpi=200)
plt.show()

# --- 安全報酬 ---
plt.figure(figsize=(10,4))
plt.plot(safe_mean, label="mean safe reward")
plt.fill_between(range(len(safe_mean)),
                 safe_mean - safe_std,
                 safe_mean + safe_std,
                 alpha=0.3)
plt.ylim(-200, 0)
plt.title(f"Safety Reward (mean ± std over {n_runs} runs)")
plt.grid()
plt.legend()
plt.savefig("safe_mean_std.png", dpi=200)
plt.show()



In [ ]:
print(f"=== Safety vs Performance Statistics ({nav_all.shape[0]} runs) ===")

print(f"Nav Reward: mean={nav_all.mean():.2f}, std={nav_all.std():.2f}")
print(f"Safe Reward: mean={safe_all.mean():.2f}, std={safe_all.std():.2f}")

print(f"h(x): mean={h_all.mean():.4f}, std={h_all.std():.4f}")
print(f"h_value: mean={hval_all.mean():.4f}, std={hval_all.std():.4f}")

print(f"m_dir: mean={mdir_all.mean():.4f}, std={mdir_all.std():.4f}")
print(f"grad_dot: mean={dot_all.mean():.4f}, std={dot_all.std():.4f}")

print(f"Violations: mean={viol_all.mean():.2f}, std={viol_all.std():.2f}")

print(f"Runtime: mean={runtimes.mean():.2f} sec, std={runtimes.std():.2f} sec")


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(hval_all.flatten(), mdir_all.flatten(), alpha=0.2, s=5)
plt.xlabel("h_value (Safety Level)")
plt.ylabel("m_dir (Safety vs Performance Mix)")
plt.title(f"m_dir vs h_value ({hval_all.shape[0]} runs × episodes)")
plt.grid()
plt.savefig("scatter_mdir_hvalue.png", dpi=200)   # ★ これを追加
plt.close() 


In [ ]:
print(f"{len(runtimes)}回分の実行時間（秒）:")
print(runtimes)

print(f"\n平均実行時間: {runtimes.mean():.2f} 秒")
print(f"最短実行時間: {runtimes.min():.2f} 秒")
print(f"最長実行時間: {runtimes.max():.2f} 秒")
print(f"標準偏差: {runtimes.std():.2f} 秒")

plt.figure(figsize=(8,4))
plt.hist(runtimes, bins=20, color='skyblue', edgecolor='black')
plt.title(f"Runtime Distribution ({len(runtimes)} runs)")
plt.xlabel("seconds")
plt.ylabel("count")
plt.grid()
plt.savefig("runtime_hist.png", dpi=200)
plt.close()


## 【付録：現在動作しないレガシーコード（隔離）】

以下のセル群（ffmpegインストール／単発シミュレーション・動画生成／動画再生）は，複数run対応の
リファクタリング以前に書かれた，単発runを前提とした可視化コードです。

`run_experiment()` 内で `agent` / `env` / `memory` がローカル変数として扱われるようになったため，
これらのセルが参照するグローバル変数 `agent` 等は学習ループ終了後には存在せず，
現状のまま実行すると `NameError` 等で失敗します。

研究アルゴリズム・実験結果には一切影響しないため削除はせず，**参考用の非推奨レガシー付録**として
ここに隔離しています。通常の実験実行では，このセクションを実行する必要はありません。

In [ ]:
!apt-get install -y ffmpeg

In [ ]:
import torch

agent.actor_net.load_state_dict(torch.load("policy_ep200.pth"))
agent.actor_net.eval()
print(">>> actor_safe_only.pth をロードしました。安全制御のみのアクターでシミュレーション開始 <<<")

import numpy as np
import matplotlib.pyplot as plt
import math
import os

# --- シミュレーション（既存コードを想定） ---
env = gym.make(gym_name, render_mode='rgb_array')
env.action_space.seed(seed)

frames = []
# Gym のバージョン差に対応して reset の戻り値を扱う
reset_ret = env.reset()
if isinstance(reset_ret, tuple) and len(reset_ret) >= 1:
    state = reset_ret[0]
else:
    state = reset_ret

obs_list = []
done = False

for step in range(200):
    # action の取得（agent の実装に依存）
    action = agent.select_action(state, evaluate=True)
    step_ret = env.step(action)
    # Gym のバージョン差に対応して unpack
    if len(step_ret) == 5:
        next_state, reward, terminated, truncated, info = step_ret
    else:
        # 古いバージョン: (next_state, reward, done, info)
        next_state, reward, done_flag, info = step_ret
        terminated = done_flag
        truncated = False

    if terminated or truncated:
        done = True

    obs_list.append(next_state)

    # render は render_mode='rgb_array' のとき配列を返す
    try:
        frame = env.render()
        frames.append(frame)
    except Exception:
        # render が使えない環境でも続行
        pass

    state = next_state
    if done:
        print(f"Episode ended at step={step}")
        break

env.close()

print("Collected observations:", len(obs_list))
if len(obs_list) > 0:
    print("Sample obs[0]:", np.asarray(obs_list[0]).shape, np.asarray(obs_list[0]))

# --- 観測から cos, sin, theta_dot を抽出（堅牢処理） ---
cos_list = []
sin_list = []
thetadot_list = []

for obs in obs_list:
    obs = np.asarray(obs, dtype=float).ravel()
    # デフォルト値
    cos_theta = 0.0; sin_theta = 0.0; theta_dot = 0.0

    if obs.size >= 3:
        cos_theta = obs[0]
        sin_theta = obs[1]
        theta_dot = obs[2]
    elif obs.size == 2:
        theta = obs[0]
        theta_dot = obs[1]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
    elif obs.size == 1:
        theta = obs[0]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
        theta_dot = 0.0

    # 正規化（数値誤差対策）
    norm_cs = math.hypot(cos_theta, sin_theta)
    if norm_cs > 1e-6:
        cos_theta /= norm_cs
        sin_theta /= norm_cs

    cos_list.append(float(cos_theta))
    sin_list.append(float(sin_theta))
    thetadot_list.append(float(theta_dot))

# --- プロット ---
steps = np.arange(len(cos_list))
L = len(steps)
x_lst = [0, L]
y_lst = [-2.0, -2.0]
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax0, ax1, ax2 = axes

ax0.plot(steps, thetadot_list, color='tab:blue')
ax0.plot(x_lst, y_lst, linestyle='dashed', color='red', label='Safety threshold −2.0')
ax0.set_ylabel('Angular velocity (θ̇)')
ax0.grid(True)
ax0.set_title('Simulation: step vs θ̇, sinθ, cosθ')

ax1.plot(steps, sin_list, color='tab:orange')
ax1.set_ylabel('sin θ')
ax1.grid(True)

ax2.plot(steps, cos_list, color='tab:green')
ax2.set_ylabel('cos θ')
ax2.set_xlabel('Step')
ax2.grid(True)

plt.tight_layout()

# 表示（Jupyter なら表示、スクリプトならファイル保存して確認）
try:
    plt.show()
except Exception:
    pass

# 画像保存（必ず保存しておく）
out_dir = "./figs"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "sim_traces.png")
fig.savefig(out_path, dpi=200, bbox_inches='tight')
print("Saved plot to:", out_path)


>>> actor_safe_only.pth をロードしました。安全制御のみのアクターでシミュレーション開始 <<<
Episode ended at step=199
Collected observations: 200
Sample obs[0]: (3,) [-0.31974062  0.9475051   0.33401   ]
Saved plot to: ./figs/sim_traces.png


In [ ]:
"""env = gym.make(gym_name, render_mode='rgb_array')  # render_modeを指定。シミュレーション結果を画像の配列形式で返します（例えば、framesとしてフレームを保存できます）。
#env = gym.make(gym_name)
env.action_space.seed(seed)

frames = []
state, _ = env.reset()
done = False
for step in range(200):
#while not done:
    if step<0:
      action = env.action_space.sample()
    else:
      action = agent.select_action(state, evaluate=True)
    next_state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
      done=True

    #print(f"state={next_state}, action={action}")
    if done:
      print(f"break, step={step}")
     # break
    frames.append(env.render())
    state=next_state
env.close()"""

'env = gym.make(gym_name, render_mode=\'rgb_array\')  # render_modeを指定。シミュレーション結果を画像の配列形式で返します（例えば、framesとしてフレームを保存できます）。\n#env = gym.make(gym_name)\nenv.action_space.seed(seed)\n\nframes = []\nstate, _ = env.reset()\ndone = False\nfor step in range(200):\n#while not done:\n    if step<0:\n      action = env.action_space.sample()\n    else:\n      action = agent.select_action(state, evaluate=True)\n    next_state, reward, terminated, truncated, _ = env.step(action)\n    if terminated or truncated:\n      done=True\n\n    #print(f"state={next_state}, action={action}")\n    if done:\n      print(f"break, step={step}")\n     # break\n    frames.append(env.render())\n    state=next_state\nenv.close()'

In [ ]:
from matplotlib import pyplot as plt
from matplotlib import animation

# 動画作成のセットアップ
fig = plt.figure()
patch = plt.imshow(frames[0], animated=True)

def update(frame):
    patch.set_data(frame)
    return patch,

ani = animation.FuncAnimation(

    fig, update, frames=frames, interval=100, blit=True
)

# 保存（mp4形式）
ani.save('pendulum_simulation.mp4', writer='ffmpeg')

In [ ]:
from IPython.display import Video

# 動画の再生
Video('pendulum_simulation.mp4', embed=True)